In [1]:
# Install sentence-transformers for vector generation and faiss-cpu for fast vector indexing
!pip install sentence-transformers faiss-cpu pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 41.0 MB/s eta 0:00:00


In [5]:
import kagglehub
import pandas as pd
import numpy as np


# Use single or double quotes around the file path
df = pd.read_csv("customer_support_tickets.csv")
df.head()
print(df.shape)

(8469, 17)


In [9]:
# Cell 3: Clean and Preprocess Text
print("Starting Preprocessing...")

# Define a function to clean and merge text fields
def preprocess_ticket(row):
    description = str(row['Ticket Description'])
    product = str(row['Product Purchased'])
    subject = str(row['Ticket Subject'])

    # 1. Clean the dynamic placeholder {product_purchased} if it exists in the text
    cleaned_desc = description.replace("{product_purchased}", product)

    # 2. Combine product name, subject, and description into a single cohesive string
    combined_text = f"Product: {product} | Subject: {subject} | Details: {cleaned_desc}"
    return combined_text

# Apply preprocessing to the entire dataset
df['Prepared_Text'] = df.apply(preprocess_ticket, axis=1)

# Keep a full clean copy of the dataframe for index mapping
df_indexed = df.copy()

print(f"Preprocessing completed! Total tickets prepared for embedding: {len(df_indexed)}")
print("\n--- Sample of Preprocessed Text ---")
print(df_indexed['Prepared_Text'].iloc[0])

Starting Preprocessing...
Preprocessing completed! Total tickets prepared for embedding: 8469

--- Sample of Preprocessed Text ---
Product: GoPro Hero | Subject: Product setup | Details: I'm having an issue with the GoPro Hero. Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website address.

Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.


In [10]:
# Cell 4: Generate Dense Embeddings using Pretrained Sentence-BERT
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading Pretrained Sentence-BERT Model ('all-MiniLM-L6-v2')...")
# This model maps sentences/paragraphs to a 384-dimensional dense vector space
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating vector embeddings for customer support tickets... (This will take around 1 minute)")
corpus_embeddings = model.encode(df_indexed['Prepared_Text'].tolist(), show_progress_bar=True)

# Convert embeddings to float32 NumPy array as required by FAISS
corpus_embeddings = np.array(corpus_embeddings).astype('float32')
print(f"Embeddings successfully generated! Shape of embedding matrix: {corpus_embeddings.shape}")

Loading Pretrained Sentence-BERT Model ('all-MiniLM-L6-v2')...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating vector embeddings for customer support tickets... (This will take around 1 minute)


Batches:   0%|          | 0/265 [00:00<?, ?it/s]

Embeddings successfully generated! Shape of embedding matrix: (8469, 384)


In [11]:
# Cell 5: Initialize FAISS Vector Index and Store Embeddings
import faiss

print("Initializing FAISS Vector Index...")

# 1. Normalize vectors to unit length (L2 norm = 1) so Inner Product matches Cosine Similarity
faiss.normalize_L2(corpus_embeddings)

# 2. Retrieve vector dimension (384 for all-MiniLM-L6-v2)
dimension = corpus_embeddings.shape[1]

# 3. Create an IndexFlatIP index (Inner Product index)
index = faiss.IndexFlatIP(dimension)

# 4. Populate the FAISS index with our normalized ticket embeddings
index.add(corpus_embeddings)

print(f"FAISS index successfully built and populated with {index.ntotal} vectors.")

Initializing FAISS Vector Index...
FAISS index successfully built and populated with 8469 vectors.


In [12]:
# Cell 6: Test Semantic Search with 10 Sample Queries (Showing Top-k Results)
print("Running Semantic Search Verification on 10 Sample User Queries...\n")

# Define 10 diverse user queries representing natural language technical/billing complaints
test_queries = [
    "How can I set up my new GoPro action camera?",
    "My smart TV screen keeps flickering and flashing.",
    "I have a laptop network issue and cannot connect to Wi-Fi.",
    "I was charged twice for my subscription and need a refund.",
    "The package arrived broken and damaged.",
    "Forgot my password and can't sign into my profile.",
    "Is this software compatible with the latest Windows version?",
    "The device battery is draining way too fast.",
    "My bluetooth headphones won't pair with my mobile phone.",
    "Where is my package? The shipping delivery is delayed."
]

# Execute semantic lookup for each query
for idx, query in enumerate(test_queries, 1):
    print(f"🔹 Query {idx}: '{query}'")

    # Encode and normalize user input query
    query_vector = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_vector)

    # Retrieve top-2 closest matches from FAISS
    k = 2
    cos_scores, matched_indices = index.search(query_vector, k)

    # Display results
    for rank in range(k):
        match_idx = matched_indices[0][rank]
        score = cos_scores[0][rank]
        matched_row = df_indexed.iloc[match_idx]

        print(f"  [Rank {rank+1}] Cosine Similarity: {score:.4f}")
        print(f"  Ticket ID: {matched_row['Ticket ID']} | Product: {matched_row['Product Purchased']} | Subject: {matched_row['Ticket Subject']}")
        print(f"  Snippet: {matched_row['Ticket Description'][:140]}...")
    print("-" * 85)

Running Semantic Search Verification on 10 Sample User Queries...

🔹 Query 1: 'How can I set up my new GoPro action camera?'
  [Rank 1] Cosine Similarity: 0.7777
  Ticket ID: 8164 | Product: GoPro Action Camera | Subject: Payment issue
  Snippet: I'm having an issue with the {product_purchased}. Please assist. We all can't do it, not with the same person.

But the problem with using t...
  [Rank 2] Cosine Similarity: 0.7418
  Ticket ID: 3916 | Product: GoPro Action Camera | Subject: Software bug
  Snippet: I'm having an issue with the {product_purchased}. Please assist.

#1: Make sure the screen is connected.

I can't see the screen. There is n...
-------------------------------------------------------------------------------------
🔹 Query 2: 'My smart TV screen keeps flickering and flashing.'
  [Rank 1] Cosine Similarity: 0.7907
  Ticket ID: 1193 | Product: LG Smart TV | Subject: Delivery problem
  Snippet: There seems to be a hardware problem with my {product_purchased}. The screen i

In [13]:
# Cell 7: Build and Launch the Interactive Gradio Web Interface
!pip install gradio -q  # Ensure latest Gradio framework is present

import gradio as gr
import pandas as pd

def semantic_search_engine(query, top_k):
    if not query.strip():
        return pd.DataFrame()

    # Preprocess and encode the live interface query
    query_vector = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_vector)

    # Query FAISS index for user-specified k results
    scores, indices = index.search(query_vector, int(top_k))

    # Extract matching records into a display dataframe
    search_results = []
    for rank in range(int(top_k)):
        idx = indices[0][rank]
        score = scores[0][rank]
        row = df_indexed.iloc[idx]

        search_results.append({
            "Rank": rank + 1,
            "Similarity Score": round(float(score), 4),
            "Product": row['Product Purchased'],
            "Subject": row['Ticket Subject'],
            "Description": row['Ticket Description'],
            "Priority": row['Ticket Priority'],
            "Status": row['Ticket Status']
        })

    return pd.DataFrame(search_results)

# Design User Interface with Gradio Blocks
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Semantic Search Engine for Customer Support Tickets")
    gr.Markdown(
        "This system maps raw customer language queries to a high-dimensional vector space via **Sentence-BERT** "
        "and queries an index using **FAISS** to find contextually identical support issues instantly."
    )

    with gr.Row():
        with gr.Column(scale=4):
            query_input = gr.Textbox(
                label="Type your Support Query or Problem Statement:",
                placeholder="e.g., Screen has glitches or display is dead...",
                lines=2
            )
        with gr.Column(scale=1):
            k_slider = gr.Slider(
                label="Matches to Return (Top-K)",
                minimum=1,
                maximum=10,
                step=1,
                value=3
            )

    search_button = gr.Button("🔍 Find Mathematically Similar Tickets", variant="primary")
    output_table = gr.Dataframe(label="Top Matching Historical Tickets")

    # Bind triggers for mouse click or enter key
    search_button.click(fn=semantic_search_engine, inputs=[query_input, k_slider], outputs=output_table)
    query_input.submit(fn=semantic_search_engine, inputs=[query_input, k_slider], outputs=output_table)

# Launch app with a share link visible within Colab notebooks
demo.launch(share=True)

/tmp/ipykernel_1441/2694522881.py:38: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://705bfeb07b02b96628.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
